In [1]:
#for (q,e) in zip(range(0,self.q), trange(self.q,desc = 'Epoch ' + str(epoch)+ ' Progress: ')):
import multiprocessing as mp

manager = mp.Manager()

grad = manager.list() # Must save as a list
grad_flat = manager().list() # Must save as a list
loss_value = mp.Array('d', [0,0]*100)

Vars = self.wrap_training_variables()

lock = mp.lock()

#loss_value = tf.constant(0,dtype = self.dtype)
                         

#for i in range(0, len(Vars)):
#    grad_shape = Vars[i].numpy().shape
#    grad.append(tf.convert_to_tensor(np.zeros(grad_shape),dtype = self.dtype))
#    grad_flat.append(tf.reshape(grad[i], [-1]))
    
for i in range(len(Vars)):
    grad_shape = Vars[i].numpy().shape
    grad.append(np.zeros(grad_shape, dtype=np.float32))  # Use NumPy array instead of TensorFlow tensor
    grad_flat.append(np.reshape(grad[i], -1))  # Flatten the gradients for shared memory
    
                

qvalue = 4
print(f"Running iteration q={qvalue} for Epoch {epoch}")

#timeloss2 = time.time()

with tf.GradientTape() as tape:
    self.set_weights(w)
    tape.watch(self.D)
    tape.watch(self.N)
    # tape.watch(self.fR)
    # tape.watch(self.Ram_res)
    #this loss function is time intensive

    timeloss2 = time.time()

    loss_value_q = self.loss(t,qvalue, FROG_0, FROG_1)

timeloss2middle = time.time()

#loss_value = loss_value + loss_value_q
#######################This is a lossy line related to gradient computation

#@tf.function
#def compute_gradients(loss_value_q):  
#    return tape.gradient(loss_value_q, self.wrap_training_variables())

#grad_q = compute_gradients(loss_value_q)

grad_q = tape.gradient(loss_value_q, self.wrap_training_variables())

timeloss3 = time.time()

with lock:  # Ensure thread-safety
    for i, g_q in zip(range(0,len(Vars)),grad_q): # summnation of gradients
        #grad_flat[i] = grad_flat[i] + tf.reshape(g_q, -1)
        grad_flat[i] += np.reshape(g_q.numpy(), -1)  # Convert to NumPy and sum


del tape

timeloss4 = time.time()

timeloss23= timeloss3-timeloss2
timeloss34= timeloss4-timeloss3

timeloss2mid2 = timeloss3-timeloss2middle
timeloss2mid1 = timeloss2middle-timeloss2

print("timelossfunctionwithmake " + str(timeloss2mid1))

timeMAKEtotal +=timeloss2mid1

print("time loss other, gradient computation" + str(timeloss2mid2))

print("computing loss uses makefrog " + str(timeloss23))
#print("computing loss uses " + str(timeloss))


print("summation of gradients " + str(timeloss34))


#grad_flat = tf.concat(grad_flat, 0)

grad[qvalue] = np.array(grad_flat)

return loss_value_q, grad_flat


TypeError: 'SyncManager' object is not callable

In [ ]:

for (q,e) in zip(range(0,self.q), trange(self.q,desc = 'Epoch ' + str(epoch)+ ' Progress: ')):

    #timeloss2 = time.time()

    with tf.GradientTape() as tape:
        self.set_weights(w)
        tape.watch(self.D)
        tape.watch(self.N)
        # tape.watch(self.fR)
        # tape.watch(self.Ram_res)
        #this loss function is time intensive

        loss_value_q = self.loss(t,q, FROG_0, FROG_1)


    loss_value = loss_value + loss_value_q
    
    #######################This is a lossy line related to gradient computation
    grad_q = tape.gradient(loss_value_q, self.wrap_training_variables())


    for i, g_q in zip(range(0,len(Vars)),grad_q): # summnation of gradients
        grad_flat[i] = grad_flat[i] + tf.reshape(g_q, -1)

    del tape



grad_flat =  tf.concat(grad_flat, 0)

return loss_value, grad_flat


In [ ]:


print(f"Running iteration q={qvalue} for Epoch {epoch}")

loss_value = 0.0  # Initialize loss
grad_flat = [tf.zeros_like(var) for var in Vars]  # Initialize gradient storage

with tf.GradientTape() as tape:
    self.set_weights(w)
    tape.watch(self.D)
    tape.watch(self.N)

    # Compute loss for the specified qvalue
    loss_value_q = self.loss(t, qvalue, FROG_0, FROG_1)

loss_value += loss_value_q  # Accumulate loss

# Compute gradients
grad_q = tape.gradient(loss_value_q, self.wrap_training_variables())

# Sum gradients into grad_flat
for i, g_q in zip(range(len(Vars)), grad_q):
    grad_flat[i] += tf.reshape(g_q, -1)

del tape  # Explicitly delete tape to free memory

# Concatenate gradient tensors
grad_flat = tf.concat(grad_flat, axis=0)

return loss_value, grad_flat
